# Study 2: Pollution dynamics and mitigation

- This notebook examines how the current four-pollutant environment produces, transports, recovers, and cleans up pollution.
- I compare the current investment settings with a more demanding candidate and then test learned policies under matched seeds. The candidate is a synthetic sensitivity, not a real-world estimate.

## How the current pollution process works

- Each region produces baseline emissions plus additional emissions linked to its EDI. Prevention reduces new emissions; cleanup removes part of accumulated pollution.
- Each pollutant has its own transport, natural recovery, and cleanup settings. The four pollutant stocks remain separate, while GPI combines their normalized loads.

In [1]:
import pandas as pd

from study_2_economic_sustainability.config import EnvironmentConfig
from study_2_economic_sustainability.train import train_agents
from study_2_economic_sustainability.evaluation import run_episode
from study_2_economic_sustainability.config import TrainingConfig
from study_2_economic_sustainability.environment import (ACTION_NAMES, EconomicSustainabilityEnvironment)

In [2]:
config = EnvironmentConfig()
EDI_LEVELS = (0, 70, 100)

### Compare the current investment settings with a candidate

In [3]:
INVESTMENT_SETTINGS = {
    "current": {
        "prevention_fractions": config.prevention_fractions,
        "prevention_costs": config.prevention_costs,
        "cleanup_capacities": config.cleanup_capacities,
        "cleanup_costs": config.cleanup_costs,
    },
    "candidate": {
        "prevention_fractions": (0.0, 0.25, 0.50, 0.60),
        "prevention_costs": config.prevention_costs,
        "cleanup_capacities": (0.0, 2.0, 5.0, 8.0),
        "cleanup_costs": config.cleanup_costs,
    },
}

- The candidate lowers the strongest prevention fraction from 0.75 to 0.60 and cleanup capacity from 10 to 8 per pollutant. Costs remain fixed in this comparison.

In [4]:
investment_rows = []

for setting_name, setting in INVESTMENT_SETTINGS.items():
    for level in range(4):
        investment_rows.append({
            "setting": setting_name,
            "investment": "prevention",
            "level": level,
            "cost": setting["prevention_costs"][level],
            "prevention_fraction": setting["prevention_fractions"][level],
            "cleanup_capacity_per_pollutant": 0,
        })
        investment_rows.append({
            "setting": setting_name,
            "investment": "cleanup",
            "level": level,
            "cost": setting["cleanup_costs"][level],
            "prevention_fraction": 0,
            "cleanup_capacity_per_pollutant": setting["cleanup_capacities"][level],
        })

display(pd.DataFrame(investment_rows).set_index(["setting", "investment", "level"]))

cost  prevention_fraction  \
setting   investment level                              
current   prevention 0       0.0                 0.00   
          cleanup    0       0.0                 0.00   
          prevention 1      10.0                 0.25   
          cleanup    1      15.0                 0.00   
          prevention 2      25.0                 0.50   
          cleanup    2      40.0                 0.00   
          prevention 3      50.0                 0.75   
          cleanup    3      80.0                 0.00   
candidate prevention 0       0.0                 0.00   
          cleanup    0       0.0                 0.00   
          prevention 1      10.0                 0.25   
          cleanup    1      15.0                 0.00   
          prevention 2      25.0                 0.50   
          cleanup    2      40.0                 0.00   
          prevention 3      50.0                 0.60   
          cleanup    3      80.0                 0.00   

                            cleanup_capacity_per_pollutant  
setting   investment level                                  
current   prevention 0                                 0.0  
          cleanup    0                                 0.0  
          prevention 1                                 0.0  
          cleanup    1                                 2.0  
          prevention 2                                 0.0  
          cleanup    2                                 5.0  
          prevention 3                                 0.0  
          cleanup    3                                10.0  
candidate prevention 0                                 0.0  
          cleanup    0                                 0.0  
          prevention 1                                 0.0  
          cleanup    1                                 2.0  
          prevention 2                                 0.0  
          cleanup    2                                 5.0  
          prevention 3                                 0.0  
          cleanup    3                                 8.0

## How much does EDI add to emissions?

- This table reports gross emissions by pollutant at selected EDI levels, then shows the remaining amount after the strongest prevention level. It does not predict future pollution stocks.

In [5]:
emission_rows = []

for setting_name, setting in INVESTMENT_SETTINGS.items():
    prevention_fraction = setting["prevention_fractions"][3]
    for edi in EDI_LEVELS:
        for pollutant in config.pollutants:
            gross = config.baseline_emissions[pollutant]
            gross += config.edi_emissions_per_point[pollutant] * edi
            emission_rows.append({
                "setting": setting_name,
                "edi": edi,
                "pollutant": pollutant,
                "gross_emissions": gross,
                "remaining_after_high_prevention": gross * (1 - prevention_fraction),
            })

display(pd.DataFrame(emission_rows).set_index(["setting", "edi", "pollutant"]))

gross_emissions  remaining_after_high_prevention
setting   edi pollutant                                                  
current   0   air                    3.0                            0.750
              water                  1.0                            0.250
              soil                   0.5                            0.125
              noise                  2.0                            0.500
          70  air                    8.6                            2.150
              water                  3.8                            0.950
              soil                   1.9                            0.475
              noise                  4.1                            1.025
          100 air                   11.0                            2.750
              water                  5.0                            1.250
              soil                   2.5                            0.625
              noise                  5.0                            1.250
candidate 0   air                    3.0                            1.200
              water                  1.0                            0.400
              soil                   0.5                            0.200
              noise                  2.0                            0.800
          70  air                    8.6                            3.440
              water                  3.8                            1.520
              soil                   1.9                            0.760
              noise                  4.1                            1.640
          100 air                   11.0                            4.400
              water                  5.0                            2.000
              soil                   2.5                            1.000
              noise                  5.0                            2.000

In [6]:
# Emissions are summed by EDI so the next table is easy to compare.
emission_totals = pd.DataFrame(emission_rows).groupby(["setting", "edi"])[["gross_emissions", "remaining_after_high_prevention"]].sum()
display(emission_totals.round(2))

gross_emissions  remaining_after_high_prevention
setting   edi                                                  
candidate 0                6.5                             2.60
          70              18.4                             7.36
          100             23.5                             9.40
current   0                6.5                             1.62
          70              18.4                             4.60
          100             23.5                             5.88

- The candidate leaves a larger share of new emissions after its strongest prevention action. Remaining emissions still enter the regional pollution process, where transport, natural recovery, and cleanup affect the resulting stocks.

## Compare investment costs with income

In [7]:
income_rows = []

for setting_name, setting in INVESTMENT_SETTINGS.items():
    for edi in EDI_LEVELS:
        income = config.income_base + config.income_per_edi * edi
        prevention_cost = setting["prevention_costs"][3]
        cleanup_cost = setting["cleanup_costs"][3]
        income_rows.append({
            "setting": setting_name,
            "edi": edi,
            "income_per_step": income,
            "high_prevention_cost": prevention_cost,
            "income_after_high_prevention": income - prevention_cost,
            "high_cleanup_cost": cleanup_cost,
            "income_after_high_cleanup": income - cleanup_cost,
        })

display(pd.DataFrame(income_rows).set_index(["setting", "edi"]))

income_per_step  high_prevention_cost  \
setting   edi                                          
current   0               20.0                  50.0   
          70              55.0                  50.0   
          100             70.0                  50.0   
candidate 0               20.0                  50.0   
          70              55.0                  50.0   
          100             70.0                  50.0   

               income_after_high_prevention  high_cleanup_cost  \
setting   edi                                                    
current   0                           -30.0               80.0   
          70                            5.0               80.0   
          100                          20.0               80.0   
candidate 0                           -30.0               80.0   
          70                            5.0               80.0   
          100                          20.0               80.0   

               income_after_high_cleanup  
setting   edi                             
current   0                        -60.0  
          70                       -25.0  
          100                      -10.0  
candidate 0                        -60.0  
          70                       -25.0  
          100                      -10.0

## How does pollutant transport affect neighboring regions?

- Each pollutant has a different transport fraction. Transported emissions leave their source; the same share of recovered pollution stocks is also transferred to neighbors.
- The table shows how five extra units of emissions from a source region are divided for each pollutant.

In [8]:
EXTRA_EMISSIONS = 5
transport_rows = []

for pollutant in config.pollutants:
    fraction = config.transport_fractions[pollutant]
    transport_rows.append({
        "pollutant": pollutant,
        "source_retains": EXTRA_EMISSIONS * (1 - fraction),
        "each_neighbor_receives": EXTRA_EMISSIONS * fraction / 2,
        "total_transferred": EXTRA_EMISSIONS * fraction,
    })

display(pd.DataFrame(transport_rows).set_index("pollutant"))

,source_retains,each_neighbor_receives,total_transferred
pollutant,,,
air,3.5,0.75,1.5
water,3.0,1.00,2.0
soil,4.5,0.25,0.5
noise,4.0,0.50,1.0


- The transferred amount is divided evenly between the source region's two neighbors. It leaves the source, so transport redistributes emissions rather than duplicating them. The same process is applied separately to each pollutant.

## Test high prevention strength on its own

In [9]:
HIGH_PREVENTION_OPTIONS = (0.40, 0.50, 0.60, 0.75)
prevention_rows = []

for prevention_fraction in HIGH_PREVENTION_OPTIONS:
    for edi in EDI_LEVELS:
        total_gross = 0
        total_remaining = 0
        for pollutant in config.pollutants:
            gross = config.baseline_emissions[pollutant]
            gross += config.edi_emissions_per_point[pollutant] * edi
            total_gross += gross
            total_remaining += gross * (1 - prevention_fraction)
        prevention_rows.append({
            "high_prevention_fraction": prevention_fraction,
            "edi": edi,
            "total_gross_emissions": total_gross,
            "remaining_emissions": total_remaining,
        })

display(pd.DataFrame(prevention_rows).set_index(["high_prevention_fraction", "edi"]))

total_gross_emissions  remaining_emissions
high_prevention_fraction edi                                            
0.40                     0                      6.5                3.900
                         70                    18.4               11.040
                         100                   23.5               14.100
0.50                     0                      6.5                3.250
                         70                    18.4                9.200
                         100                   23.5               11.750
0.60                     0                      6.5                2.600
                         70                    18.4                7.360
                         100                   23.5                9.400
0.75                     0                      6.5                1.625
                         70                    18.4                4.600
                         100                   23.5                5.875

- Stronger prevention lowers new emissions at every EDI level, but leaves a share of baseline emissions. It does not remove pollution already in the regional stocks.

## Check how cleanup changes with contamination


In [10]:
cleanup_rows = []

for pollutant in config.pollutants:
    for load_fraction in (0.25, 0.50, 1.0):
        pollution_load = config.pollution_limits[pollutant] * load_fraction
        efficiency = 1 - config.cleanup_difficulty[pollutant] * load_fraction
        efficiency = max(config.minimum_cleanup_efficiency, efficiency)
        cleanup_capacity = config.cleanup_capacities[3]
        pollution_removed = min(cleanup_capacity * efficiency, pollution_load)
        cleanup_rows.append({
            "pollutant": pollutant,
            "pollution_load_fraction": load_fraction,
            "cleanup_efficiency": efficiency,
            "pollution_removed": pollution_removed,
        })

display(pd.DataFrame(cleanup_rows).set_index(["pollutant", "pollution_load_fraction"]))

cleanup_efficiency  pollution_removed
pollutant pollution_load_fraction                                       
air       0.25                                 1.0000             10.000
          0.50                                 1.0000             10.000
          1.00                                 1.0000             10.000
water     0.25                                 0.8750              8.750
          0.50                                 0.7500              7.500
          1.00                                 0.5000              5.000
soil      0.25                                 0.8375              8.375
          0.50                                 0.6750              6.750
          1.00                                 0.3500              3.500
noise     0.25                                 1.0000             10.000
          0.50                                 1.0000             10.000
          1.00                                 1.0000             10.000

- Cleanup efficiency falls as contamination rises for pollutants with a nonzero difficulty setting. The configured minimum efficiency prevents cleanup from reaching zero effectiveness.

## Explore a range of environmental investment costs

- Compare prevention and cleanup costs with income at several development levels.

In [11]:
INVESTMENT_COST_MULTIPLIERS = (0.5, 1.0, 1.5, 2.0)

cost_rows = []
for cost_multiplier in INVESTMENT_COST_MULTIPLIERS:
    for investment_name, costs in (("prevention", config.prevention_costs),("cleanup", config.cleanup_costs)):
        for level in range(1, len(costs)):
            action_cost = costs[level] * cost_multiplier
            for edi in (0, 70, 100):
                income = config.income_base + config.income_per_edi * edi
                cost_rows.append({
                    "cost_multiplier": cost_multiplier,
                    "investment": investment_name,
                    "level": level,
                    "edi": edi,
                    "investment_cost": action_cost,
                    "income_per_step": income,
                    "income_after_cost": income - action_cost,
                })

display(pd.DataFrame(cost_rows).set_index(["cost_multiplier", "investment", "level", "edi"]))

investment_cost  income_per_step  \
cost_multiplier investment level edi                                     
0.5             prevention 1     0                5.0             20.0   
                                 70               5.0             55.0   
                                 100              5.0             70.0   
                           2     0               12.5             20.0   
                                 70              12.5             55.0   
...                                               ...              ...   
2.0             cleanup    2     70              80.0             55.0   
                                 100             80.0             70.0   
                           3     0              160.0             20.0   
                                 70             160.0             55.0   
                                 100            160.0             70.0   

                                      income_after_cost  
cost_multiplier investment level edi                     
0.5             prevention 1     0                 15.0  
                                 70                50.0  
                                 100               65.0  
                           2     0                  7.5  
                                 70                42.5  
...                                                 ...  
2.0             cleanup    2     70               -25.0  
                                 100              -10.0  
                           3     0               -140.0  
                                 70              -105.0  
                                 100              -90.0  

[72 rows x 3 columns]

- Compare the table's costs with income to see when an investment may require saved currency. Actual affordability also depends on development costs, starting balance, and the sequence of actions.

## Pilot learned policies under candidate settings

- This pilot compares four synthetic combinations of prevention strength and investment cost. The pollutant-specific transport and recovery settings stay at their current values.
- Each setting uses the same five training seeds and 20 held-out starts. These are exploratory comparisons, not calibrated recommendations.

In [12]:
PILOT_TRAINING_SEEDS = (11, 23, 37, 41, 53)
PILOT_EVALUATION_SEEDS = tuple(range(100_000, 100_020))
PILOT_TRAINING_EPISODES = 2_000
PILOT_REWARD_MODE = "mixed"
pilot_training_config = TrainingConfig()
pilot_training_config.training_episodes = PILOT_TRAINING_EPISODES

In [13]:
PILOT_SETTINGS = {
    "current": {"prevention_scale": 1.0, "cost_scale": 1.0},
    "weaker_prevention": {"prevention_scale": 0.8, "cost_scale": 1.0},
    "higher_cost": {"prevention_scale": 1.0, "cost_scale": 1.5},
    "weaker_prevention_and_higher_cost": {
        "prevention_scale": 0.8, "cost_scale": 1.5
    },
}

def make_pilot_config(setting):
    pilot_config = EnvironmentConfig()
    prevention_fractions = []

    for fraction in config.prevention_fractions:
        prevention_fractions.append(fraction * setting["prevention_scale"])

    pilot_config.prevention_fractions = tuple(prevention_fractions)
    prevention_costs = []
    cleanup_costs = []

    for cost in config.prevention_costs:
        prevention_costs.append(cost * setting["cost_scale"])

    for cost in config.cleanup_costs:
        cleanup_costs.append(cost * setting["cost_scale"])

    pilot_config.prevention_costs = tuple(prevention_costs)
    pilot_config.cleanup_costs = tuple(cleanup_costs)
    transport_multiplier = setting.get("transport_multiplier", 1.0)

    for pollutant in config.pollutants:
        pilot_config.transport_fractions[pollutant] = (config.transport_fractions[pollutant] * transport_multiplier)
    return pilot_config

In [14]:
def pilot_rewards(environment, info, terminated):
    local_rewards = []

    for region in range(4):
        income_bonus = (info["income"][region] - environment.config.income_base)
        income_bonus /= environment.config.ongoing_income_reward_scale
        economic_reward = info["development_gains"][region] + income_bonus
        reward = environment.config.economy_weight * economic_reward
        reward -= environment.config.pollution_weight * info["pollution_damage"][region]

        if not info["valid_actions"][region]:
            reward -= environment.config.invalid_action_penalty
        if terminated:
            reward -= environment.config.failure_penalty
        local_rewards.append(reward)

    shared_reward = sum(local_rewards) / 4
    mixed_rewards = []

    for reward in local_rewards:
        mixed_rewards.append(0.7 * reward + 0.3 * shared_reward)
    return tuple(mixed_rewards)

In [15]:
def train_pilot_policy(setting, training_seed):
    pilot_config = make_pilot_config(setting)

    def reward_transform(environment, rewards, info, terminated):
        return pilot_rewards(environment, info, terminated)

    return train_agents(PILOT_REWARD_MODE, training_seed, pilot_config, pilot_training_config, reward_transform)

In [16]:
pilot_policies = {}
for setting_name, setting in PILOT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        pilot_policies[(setting_name, training_seed)] = train_pilot_policy(setting, training_seed)
    print(f"Trained {setting_name} across five seeds")

Trained current across five seeds
Trained weaker_prevention across five seeds
Trained higher_cost across five seeds
Trained weaker_prevention_and_higher_cost across five seeds


## Evaluate on the same held-out starts

In [17]:
def create_pilot_totals():
    return {
        "action_counts": [0] * len(ACTION_NAMES),
        "pollution_values": [],
        "transport_inflow_values": [],
        "regional_spreads": [],
        "remaining_emissions": 0.0,
        "prevention_spending": 0.0,
        "cleanup_spending": 0.0,
    }

In [18]:
def record_pilot_step(totals, info):
    for pollutant in info["remaining_emissions"]:
        totals["remaining_emissions"] += sum(info["remaining_emissions"][pollutant])
        totals["pollution_values"].extend(info["pollutant_loads"][pollutant])
        totals["transport_inflow_values"].extend(info["incoming_emissions"][pollutant])
        totals["transport_inflow_values"].extend(info["incoming_pollution"][pollutant])

    totals["prevention_spending"] += sum(info["prevention_costs_paid"])
    totals["cleanup_spending"] += sum(info["cleanup_costs_paid"])
    regional_indices = info["regional_pollution_indices"]
    regional_spread = max(regional_indices) - min(regional_indices)
    totals["regional_spreads"].append(regional_spread)

In [19]:
def summarize_pilot_episode(environment, totals, setting_name, seed, training_seed, failed):
    pollution_values = totals["pollution_values"]
    transport_inflow_values = totals["transport_inflow_values"]
    regional_spreads = totals["regional_spreads"]
    episode_region_steps = 4 * environment.step_count

    row = {
        "setting": setting_name,
        "training_seed": training_seed,
        "evaluation_seed": seed,
        "mean_pollution": sum(pollution_values) / len(pollution_values),
        "peak_pollution": max(pollution_values),
        "remaining_emissions": totals["remaining_emissions"],
        "prevention_spending": totals["prevention_spending"],
        "cleanup_spending": totals["cleanup_spending"],
        "final_edi": sum(environment.development_indices) / 4,
        "final_currency": sum(environment.currency_balances) / 4,
        "failed": failed,
        "mean_transport_inflow": sum(transport_inflow_values) / len(transport_inflow_values),
        "mean_regional_spread": sum(regional_spreads) / len(regional_spreads),
    }

    for action_id, action_name in enumerate(ACTION_NAMES):
        row[f"{action_name}_share"] = (totals["action_counts"][action_id] / episode_region_steps)
    return row

In [20]:
def evaluate_pilot_episode(agents, setting, seed, setting_name, training_seed):
    pilot_config = make_pilot_config(setting)
    environment = EconomicSustainabilityEnvironment(PILOT_REWARD_MODE, pilot_config)
    totals = create_pilot_totals()
    transitions, terminated, truncated = run_episode(environment, agents, seed)

    for transition in transitions:
        for action in transition["actions"]:
            totals["action_counts"][action] += 1
        record_pilot_step(totals, transition["info"])

    return summarize_pilot_episode(environment, totals, setting_name, seed, training_seed, terminated)

In [21]:
pilot_episode_rows = []
for setting_name, setting in PILOT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        agents = pilot_policies[(setting_name, training_seed)]
        for evaluation_seed in PILOT_EVALUATION_SEEDS:
            pilot_episode_rows.append(evaluate_pilot_episode(agents, setting, evaluation_seed, setting_name, training_seed))

pilot_episodes = pd.DataFrame(pilot_episode_rows)

In [ ]:
PILOT_METRICS = [
    "mean_pollution", "peak_pollution", "remaining_emissions",
    "prevention_spending", "cleanup_spending", "final_edi",
    "final_currency", "failed", "mean_transport_inflow",
    "mean_regional_spread",
    "maintain_none_share", "develop_none_share",
    "maintain_prevent_high_share", "maintain_clean_high_share",
    ]

pilot_seed_results = pilot_episodes.groupby(["setting", "training_seed"])[PILOT_METRICS].mean()
setting_results = pilot_seed_results.groupby(level="setting")

In [22]:
display(setting_results[["mean_pollution", "peak_pollution", "remaining_emissions", "prevention_spending", "cleanup_spending","final_edi"]].agg(["mean", "std"]).round(2))

mean_pollution       peak_pollution         \
                                            mean   std           mean    std   
setting                                                                        
current                                    14.93  2.29          47.31   8.70   
higher_cost                                13.20  2.86          38.32   4.29   
weaker_prevention                           8.80  6.19          33.18  14.28   
weaker_prevention_and_higher_cost          13.92  4.14          37.88   6.57   

                                  remaining_emissions           \
                                                 mean      std   
setting                                                          
current                                      12083.22  1850.05   
higher_cost                                  11578.30  1991.99   
weaker_prevention                            12948.52   454.03   
weaker_prevention_and_higher_cost            13431.78  1318.49   

                                  prevention_spending            \
                                                 mean       std   
setting                                                           
current                                      10424.35   6232.72   
higher_cost                                  19752.00  11058.00   
weaker_prevention                             5487.15   2755.48   
weaker_prevention_and_higher_cost            13334.10   3609.32   

                                  cleanup_spending          final_edi         
                                              mean      std      mean    std  
setting                                                                       
current                                    8749.40  3052.63     81.92   8.17  
higher_cost                               13237.28  5867.28     90.18   9.49  
weaker_prevention                         19517.80  9938.21     72.84   6.34  
weaker_prevention_and_higher_cost         15998.10  2658.90     84.08  11.30

In [23]:
display(setting_results[["final_currency", "failed", "mean_transport_inflow"]].agg(["mean", "std"]).round(2))

final_currency          failed       \
                                            mean      std   mean  std   
setting                                                                 
current                                  5847.37  1086.06    0.0  0.0   
higher_cost                              2509.42  1321.30    0.0  0.0   
weaker_prevention                        4099.39  2438.87    0.0  0.0   
weaker_prevention_and_higher_cost        3332.36  1655.20    0.0  0.0   

                                  mean_transport_inflow        
                                                   mean   std  
setting                                                        
current                                            1.85  0.26  
higher_cost                                        1.69  0.20  
weaker_prevention                                  1.33  0.56  
weaker_prevention_and_higher_cost                  1.90  0.45

In [24]:
display(setting_results[["develop_none_share", "maintain_prevent_high_share", "maintain_clean_high_share"]].agg(["mean", "std"]).round(2))

develop_none_share        \
                                                mean   std   
setting                                                      
current                                         0.02  0.01   
higher_cost                                     0.03  0.02   
weaker_prevention                               0.02  0.01   
weaker_prevention_and_higher_cost               0.01  0.01   

                                  maintain_prevent_high_share        \
                                                         mean   std   
setting                                                               
current                                                  0.14  0.18   
higher_cost                                              0.22  0.19   
weaker_prevention                                        0.05  0.08   
weaker_prevention_and_higher_cost                        0.16  0.08   

                                  maintain_clean_high_share        
                                                       mean   std  
setting                                                            
current                                                0.06  0.06  
higher_cost                                            0.03  0.03  
weaker_prevention                                      0.22  0.15  
weaker_prevention_and_higher_cost                      0.04  0.06

## What the pilot suggests

- Across the 20 held-out starts per training seed, none of the four settings failed. The higher-cost setting had the highest mean final EDI (90.2) and lower mean GPI (13.2) than the current setting (EDI 81.9; GPI 14.9).
- Weaker prevention lowered prevention spending and development, while cleanup spending rose. These are learned policies retrained for each setting, so the results show how the agents adapted; they do not isolate a single parameter's effect. This pilot does not establish that any setting is realistic or optimal.

## Compare pollutant transport levels

- Here I vary the configured pollutant transport fractions while holding the other current environmental settings fixed. The multipliers are synthetic sensitivities, not real-world estimates.

In [25]:
TRANSPORT_SETTINGS = {}
for transport_multiplier in (0.0, 1.0, 1.5):
    setting_name = f"transport_{transport_multiplier:.1f}x"
    TRANSPORT_SETTINGS[setting_name] = {"prevention_scale": 1.0, "cost_scale": 1.0, "transport_multiplier": transport_multiplier}

In [26]:
transport_policies = {}
for setting_name, setting in TRANSPORT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        transport_policies[(setting_name, training_seed)] = train_pilot_policy(setting, training_seed)
    print(f"Trained {setting_name} across five seeds")

Trained transport_0.0x across five seeds
Trained transport_1.0x across five seeds
Trained transport_1.5x across five seeds


### Evaluate the transport policies

In [27]:
transport_rows = []
for setting_name, setting in TRANSPORT_SETTINGS.items():
    for training_seed in PILOT_TRAINING_SEEDS:
        agents = transport_policies[(setting_name, training_seed)]
        for evaluation_seed in PILOT_EVALUATION_SEEDS:
            transport_rows.append(evaluate_pilot_episode(agents, setting, evaluation_seed, setting_name, training_seed))

transport_episodes = pd.DataFrame(transport_rows)
transport_seed_results = transport_episodes.groupby(["setting", "training_seed"])[PILOT_METRICS + ["mean_regional_spread"]].mean()
transport_summary = transport_seed_results.groupby(level="setting")

In [28]:
display(transport_summary[["mean_pollution", "peak_pollution", "remaining_emissions", "prevention_spending", "cleanup_spending", "final_edi", "failed"]].agg(["mean", "std"]).round(2))

mean_pollution       peak_pollution        remaining_emissions  \
                         mean   std           mean    std                mean   
setting                                                                         
transport_0.0x          13.79  3.02          60.52  14.77            11864.55   
transport_1.0x          14.93  2.29          47.31   8.70            12083.22   
transport_1.5x          11.78  7.97          45.16  33.11            12496.02   

                        prevention_spending          cleanup_spending  \
                    std                mean      std             mean   
setting                                                                 
transport_0.0x  2947.40            10915.05  9708.54         12770.55   
transport_1.0x  1850.05            10424.35  6232.72          8749.40   
transport_1.5x  2187.46             7477.30  8479.06         14436.15   

                         final_edi        failed       
                     std      mean    std   mean  std  
setting                                                
transport_0.0x   4700.60     79.90   9.28   0.00  0.0  
transport_1.0x   3052.63     81.92   8.17   0.00  0.0  
transport_1.5x  10381.61     80.39  10.27   0.18  0.4

In [29]:
display(transport_summary[["mean_transport_inflow", "mean_regional_spread"]].agg(["mean", "std"]).round(2))

mean_transport_inflow       mean_regional_spread               \
                                mean   std                 mean   std   mean   
setting                                                                        
transport_0.0x                  0.00  0.00                17.93  6.56  17.93   
transport_1.0x                  1.85  0.26                 8.28  3.54   8.28   
transport_1.5x                  2.39  1.08                 4.34  0.76   4.34   

                      
                 std  
setting               
transport_0.0x  6.56  
transport_1.0x  3.54  
transport_1.5x  0.76

In [30]:
display(transport_summary[["maintain_prevent_high_share", "maintain_clean_high_share"]].agg(["mean", "std"]).round(2))

maintain_prevent_high_share       maintain_clean_high_share  \
                                      mean   std                      mean   
setting                                                                      
transport_0.0x                        0.23  0.26                      0.10   
transport_1.0x                        0.14  0.18                      0.06   
transport_1.5x                        0.12  0.16                      0.12   

                      
                 std  
setting               
transport_0.0x  0.09  
transport_1.0x  0.06  
transport_1.5x  0.14

## What this comparison suggests

- Stronger transport reduced the spread in regional GPI (about 17.9 with no transport, 8.3 at the current setting, and 4.3 at 1.5×). The 1.5× setting also had an 18% failure rate, despite a lower mean GPI.
- GPI alone can hide a pollutant reaching its own failure limit. Because each setting is trained separately, these results include policy adaptation and do not isolate transport's causal effect.